In [ ]:
from pathlib import Path
import os
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(REPO_ROOT / "data"))).expanduser().resolve()
REQUIRED_FILES = ['Non_Parametric_combined.csv', 'ARJ SIR DATASET.xlsx', 'runs_test_critical_values_complete.csv', 'z_table_complete.csv', 'p_critical_values.csv']
missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Place the required datasets in {DATA_DIR}: {missing}")


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Step 1: Load and prepare the training dataset
train_df = pd.read_csv(str(DATA_DIR / "Non_Parametric_combined.csv"), header=None, names=["Question", "Label"])
X = train_df["Question"].fillna('')
y = train_df["Label"].fillna(train_df["Label"].mode()[0])

# Step 2: Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=16)

# Step 3: TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Step 4: Train model
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_tfidf, y_train)

# Step 5: Model Evaluation
y_pred = dt_model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Step 6: Load Excel file for SR NO-based question prediction
excel_df = pd.read_excel(str(DATA_DIR / "ARJ SIR DATASET.xlsx"))
excel_df['SR NO'] = excel_df['SR NO'].ffill()
excel_df['Question'] = excel_df['Question'].ffill()
questions_df = excel_df.groupby('SR NO').first().reset_index()

# Step 7: Function to predict test type from SR NO
def predict_test_by_sr_no(sr_no):
    try:
        sr_no = float(sr_no)
        question = questions_df.loc[questions_df['SR NO'] == sr_no, 'Question'].values[0]
        question_tfidf = vectorizer.transform([question])
        predicted_label = dt_model.predict(question_tfidf)[0]
        return f"Predicted test for SR NO {int(sr_no)}: {predicted_label}"
    except IndexError:
        return "SR NO not found in the dataset."
    except Exception as e:
        return f"Error: {str(e)}"

# Step 8: Use it
sr_no_input = input("Enter SR NO: ")
print(predict_test_by_sr_no(sr_no_input))


In [ ]:
#RUNS
import pandas as pd
import numpy as np
import re

# Load the datasets (update paths if needed)
df = pd.read_excel("/content/ARJ SIR DATASET (1).xlsx")
critical_df = pd.read_csv("/content/runs_test_critical_values_complete (1).csv")
z_table_df = pd.read_csv("/content/z_table_complete (1).csv")

# ✅ Clean column names
df.columns = df.columns.str.strip()
critical_df.columns = critical_df.columns.str.strip()
z_table_df.columns = z_table_df.columns.str.strip()

def extract_sequence(sequence_text):
    sequence_text = sequence_text.strip()
    numbers = re.findall(r'\d+\.?\d*', sequence_text)
    if numbers:
        numbers = list(map(float, numbers))
        return ['H' if num > 7 else 'L' for num in numbers]
    categorical = re.findall(r'[A-Za-z]', sequence_text)
    if categorical:
        return [char.upper() for char in categorical]
    return None

def count_runs(sequence):
    runs = 1
    for i in range(1, len(sequence)):
        if sequence[i] != sequence[i - 1]:
            runs += 1
    return runs

def get_critical_values(n1, n2):
    try:
        col_name = str(n2)
        if n1 not in critical_df['n1/n2'].values:
            return None, None
        lower_row = critical_df[(critical_df['n1/n2'] == n1) & (critical_df['Table'] == 'Lower Tail')]
        upper_row = critical_df[(critical_df['n1/n2'] == n1) & (critical_df['Table'] == 'Upper Tail')]
        if lower_row.empty or upper_row.empty or col_name not in critical_df.columns:
            return None, None
        R_low = int(lower_row[col_name].values[0])
        R_high = int(upper_row[col_name].values[0])
        return R_low, R_high
    except:
        return None, None

def get_z_critical(alpha):
    try:
        alpha_half = alpha / 2
        target_prob = 0.5 - alpha_half
        closest_diff = float('inf')
        closest_z = None
        for row_z in z_table_df['z']:
            for col in z_table_df.columns[1:]:
                prob = z_table_df.loc[z_table_df['z'] == row_z, col].values[0]
                if isinstance(prob, str):
                    prob = float(prob)
                diff = abs(prob - target_prob)
                if diff < closest_diff:
                    closest_diff = diff
                    closest_z = float(row_z) + float(col)
        return closest_z
    except:
        return None

def runs_test_solver(sr_no):
    row = df[df['SR NO'] == sr_no]
    if row.empty:
        print(f"❌ SR NO {sr_no} not found.")
        return
    question_text = row['Question'].values[0]
    sequence_text = row['Data'].values[0]
    alpha_value = row['Alpha value'].values[0]

    print(f"\n🔹 SR NO {sr_no}: {question_text}")
    sequence = extract_sequence(sequence_text)
    if not sequence or len(set(sequence)) != 2:
        print("❌ Error: Invalid or non-binary sequence.")
        return

    unique_vals = list(set(sequence))
    n1 = sequence.count(unique_vals[0])
    n2 = sequence.count(unique_vals[1])
    R = count_runs(sequence)

    print(f"📊 n1 ({unique_vals[0]}) = {n1}, n2 ({unique_vals[1]}) = {n2}")
    print(f"🔢 Observed Runs (R) = {R}")

    if n1 + n2 <= 20:
        print("📏 Small Sample Test")
        R_low, R_high = get_critical_values(n1, n2)
        if R_low is None or R_high is None:
            print("❌ Critical values not found.")
            return
        print(f"📉 Critical Values: R_low = {R_low}, R_high = {R_high}")
        if R < R_low or R > R_high:
            print("❌ Reject H0 (Not random)")
        else:
            print("✅ Fail to reject H0 (Random)")
    else:
        print("📏 Large Sample Test")
        mean_R = ((2 * n1 * n2) / (n1 + n2)) + 1
        std_dev_R = np.sqrt((2 * n1 * n2 * (2 * n1 * n2 - n1 - n2)) / (((n1 + n2)**2) * (n1 + n2 - 1)))
        z = (R - mean_R) / std_dev_R
        z_critical = get_z_critical(float(alpha_value))

        print(f"📊 Mean (μR) = {mean_R:.3f}")
        print(f"📉 Std Dev (σR) = {std_dev_R:.3f}")
        print(f"📈 Z = {z:.3f}, Z_critical = ±{z_critical:.3f}")
        if abs(z) > abs(z_critical):
            print("❌ Reject H0 (Not random)")
        else:
            print("✅ Fail to reject H0 (Random)")

# 🔹 Take user input for SR NO
try:
    sr_no_input = int(input("Enter SR NO to run the Runs Test: "))
    runs_test_solver(sr_no_input)
except ValueError:
    print("❌ Invalid input. Please enter a valid SR NO (integer).")


In [ ]:
#WILCOXON
import pandas as pd
import numpy as np
from scipy.stats import norm

# Load and structure the data
def load_structured_data(file_path):
    # Use pd.read_excel to read Excel files
    df = pd.read_excel(file_path)
    structured_data = []

    i = 0
    while i < len(df):
        row = df.iloc[i]
        if not pd.isna(row[0]) and isinstance(row[0], (int, float)):
            serial_no = row[0]
            question_text = row[1]
            alpha_value = row[6] if len(row) > 6 and not pd.isna(row[6]) else None
            data_cols = []
            for j in range(2, len(row)):
                if not pd.isna(row[j]) and isinstance(row[j], str):
                    data_cols.append((j, row[j]))
            data_rows = []
            next_row = i + 1
            while next_row < len(df) and pd.isna(df.iloc[next_row, 0]):
                data_row = {}
                for col_idx, col_name in data_cols:
                    if col_idx < len(df.columns) and not pd.isna(df.iloc[next_row, col_idx]):
                        data_row[col_name] = df.iloc[next_row, col_idx]
                if data_row:
                    data_rows.append(data_row)
                next_row += 1
            if data_rows:
                data_df = pd.DataFrame(data_rows)
                data_df = data_df.apply(pd.to_numeric, errors='ignore')
                structured_data.append({
                    'Serial Number': serial_no,
                    'Question': question_text,
                    'Data': data_df,
                    'Alpha Value': alpha_value
                })
            i = next_row - 1
        i += 1
    return structured_data

def infer_hypotheses(question_text):
    q = question_text.lower()
    if "greater than" in q or "at least" in q:
        return 0, "greater", "H₀: Median ≤ baseline\nH₁: Median > baseline"
    elif "less than" in q:
        return 0, "less", "H₀: Median ≥ baseline\nH₁: Median < baseline"
    elif "different" in q or "difference" in q or "two-sided" in q:
        return 0, "two-sided", "H₀: Median = baseline\nH₁: Median ≠ baseline"
    elif "effective" in q or "not effective" in q:
        return 0, "two-sided", "H₀: Not effective\nH₁: Effective"
    else:
        return 0, "two-sided", "H₀: No effect\nH₁: There is an effect"

def paired_signed_rank_test(data1, data2):
    diffs = [d1 - d2 for d1, d2 in zip(data1, data2)]
    non_zero = [(i, d) for i, d in enumerate(diffs) if d != 0]
    if not non_zero:
        raise ValueError("All paired differences are zero.")

    abs_diff = [(i, abs(d)) for i, d in non_zero]
    abs_diff.sort(key=lambda x: x[1])
    ranks = [0] * len(diffs)
    i = 0
    current_rank = 1
    while i < len(abs_diff):
        tie_val = abs_diff[i][1]
        tie_group = [abs_diff[i]]
        j = i + 1
        while j < len(abs_diff) and abs_diff[j][1] == tie_val:
            tie_group.append(abs_diff[j])
            j += 1
        avg_rank = sum(range(current_rank, current_rank + len(tie_group))) / len(tie_group)
        for idx, _ in tie_group:
            ranks[idx] = avg_rank
        current_rank += len(tie_group)
        i = j
    signed_ranks = [np.sign(diffs[i]) * ranks[i] for i in range(len(diffs))]
    T_plus = sum(r for r in signed_ranks if r > 0)
    T_minus = sum(-r for r in signed_ranks if r < 0)
    T = min(T_plus, T_minus)
    return diffs, signed_ranks, T, T_plus, T_minus

def signed_rank_test(data, median):
    diffs = [x - median for x in data]
    non_zero = [(i, d) for i, d in enumerate(diffs) if d != 0]
    if not non_zero:
        raise ValueError("All values equal to the hypothesized median.")
    abs_diff = [(i, abs(d)) for i, d in non_zero]
    abs_diff.sort(key=lambda x: x[1])
    ranks = [0] * len(diffs)
    i = 0
    current_rank = 1
    while i < len(abs_diff):
        tie_val = abs_diff[i][1]
        tie_group = [abs_diff[i]]
        j = i + 1
        while j < len(abs_diff) and abs_diff[j][1] == tie_val:
            tie_group.append(abs_diff[j])
            j += 1
        avg_rank = sum(range(current_rank, current_rank + len(tie_group))) / len(tie_group)
        for idx, _ in tie_group:
            ranks[idx] = avg_rank
        current_rank += len(tie_group)
        i = j
    signed_ranks = [np.sign(diffs[i]) * ranks[i] for i in range(len(diffs))]
    T_plus = sum(r for r in signed_ranks if r > 0)
    T_minus = sum(-r for r in signed_ranks if r < 0)
    T = min(T_plus, T_minus)
    return diffs, signed_ranks, T, T_plus, T_minus

def run_wilcoxon_test(entry):
    print(f"\n🔢 Serial No: {entry['Serial Number']}")
    print(f"📘 Question: {entry['Question']}")
    print(f"🎯 Overriding Alpha Value to 0.05")
    entry['Alpha Value'] = 0.05

    data = entry['Data']
    numeric_cols = data.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print("❌ No numeric columns found.")
        return

    if len(numeric_cols) == 3:
        print(f"\n📈 Using columns: {numeric_cols[1]} and {numeric_cols[2]} for paired test")
        values1 = data[numeric_cols[1]].dropna().astype(float).tolist()
        values2 = data[numeric_cols[2]].dropna().astype(float).tolist()
        min_len = min(len(values1), len(values2))
        values1 = values1[:min_len]
        values2 = values2[:min_len]
        print(f"📊 Sample Size: {min_len} pairs")
        median, tail, hypotheses = infer_hypotheses(entry['Question'])
        hypotheses = "H₀: No difference between paired samples\nH₁: There is a difference between paired samples"
        if "greater" in tail:
            hypotheses = "H₀: Second sample ≤ First sample\nH₁: Second sample > First sample"
        elif "less" in tail:
            hypotheses = "H₀: Second sample ≥ First sample\nH₁: Second sample < First sample"
        print("\nSTEP 1: Define Hypotheses")
        print(hypotheses)
        print("\nSTEP 2: Wilcoxon Signed-Rank Test (Paired)")
        diffs, signed_ranks, T, T_plus, T_minus = paired_signed_rank_test(values1, values2)
    else:
        test_col = numeric_cols[-1]
        values = data[test_col].dropna().astype(float).tolist()
        print(f"\n📈 Data Column Used: {test_col}")
        print(f"📊 Sample Size: {len(values)}")
        baseline_val = 0
        if "8 days" in entry['Question']:
            baseline_val = 8
        elif "5 days" in entry['Question']:
            baseline_val = 5
        median, tail, hypotheses = infer_hypotheses(entry['Question'])
        median = baseline_val
        print("\nSTEP 1: Define Hypotheses")
        print(hypotheses)
        print("\nSTEP 2: Wilcoxon Signed-Rank Test (One Sample)")
        diffs, signed_ranks, T, T_plus, T_minus = signed_rank_test(values, median)

    n = sum(1 for d in diffs if d != 0)
    print("\nSTEP 3: Differences and Signed Ranks")
    for i, (diff, rank) in enumerate(zip(diffs, signed_ranks)):
        if diff != 0:
            print(f"Pair {i+1}: Diff = {diff:.2f}, Rank = {rank:.2f}")
    print(f"\nSTEP 4: T+ = {T_plus:.2f}, T- = {T_minus:.2f}")
    print(f"Test Statistic (T) = min(T+, T-) = {T:.2f}")

    if n <= 25:
        print("\nSTEP 5: Small Sample Test")
        alpha = float(entry['Alpha Value'])

                # Expanded critical values for small samples and multiple alpha levels
        critical_values = {
            "one-tailed": {
                0.10: {
                    5: 0, 6: 2, 7: 3, 8: 5, 9: 7, 10: 10, 11: 13, 12: 16, 13: 20,
                    14: 24, 15: 29, 16: 34, 17: 39, 18: 45, 19: 51, 20: 58, 21: 65,
                    22: 72, 23: 80, 24: 88, 25: 96
                },
                0.05: {
                    5: 0, 6: 2, 7: 3, 8: 6, 9: 8, 10: 11, 11: 14, 12: 17, 13: 21,
                    14: 26, 15: 30, 16: 36, 17: 41, 18: 47, 19: 54, 20: 60, 21: 68,
                    22: 75, 23: 83, 24: 92, 25: 101
                },
                0.025: {
                    6: 0, 7: 1, 8: 3, 9: 5, 10: 8, 11: 11, 12: 14, 13: 18, 14: 23,
                    15: 28, 16: 33, 17: 39, 18: 45, 19: 51, 20: 58, 21: 66, 22: 74,
                    23: 82, 24: 91, 25: 100
                },
                0.01: {
                    6: 0, 7: 1, 8: 2, 9: 4, 10: 6, 11: 9, 12: 12, 13: 16, 14: 21,
                    15: 26, 16: 31, 17: 37, 18: 43, 19: 49, 20: 56, 21: 64, 22: 72,
                    23: 81, 24: 90, 25: 99
                }
            },
            "two-tailed": {
                0.10: {
                    5: 0, 6: 1, 7: 2, 8: 4, 9: 6, 10: 9, 11: 12, 12: 15, 13: 19,
                    14: 24, 15: 28, 16: 34, 17: 39, 18: 45, 19: 51, 20: 58, 21: 66,
                    22: 73, 23: 81, 24: 90, 25: 99
                },
                0.05: {
                    5: 0, 6: 2, 7: 4, 8: 6, 9: 8, 10: 11, 11: 14, 12: 17, 13: 21,
                    14: 26, 15: 30, 16: 36, 17: 41, 18: 47, 19: 54, 20: 60, 21: 68,
                    22: 75, 23: 83, 24: 92, 25: 101
                },
                0.025: {
                    6: 0, 7: 1, 8: 3, 9: 5, 10: 8, 11: 11, 12: 14, 13: 18, 14: 23,
                    15: 28, 16: 33, 17: 39, 18: 45, 19: 51, 20: 58, 21: 66, 22: 74,
                    23: 82, 24: 91, 25: 100
                },
                0.01: {
                    6: 0, 7: 1, 8: 2, 9: 4, 10: 6, 11: 9, 12: 12, 13: 16, 14: 21,
                    15: 26, 16: 31, 17: 37, 18: 43, 19: 49, 20: 56, 21: 64, 22: 72,
                    23: 81, 24: 90, 25: 99
                }
            }
        }

        tail_type = "two-tailed" if tail == "two-sided" else "one-tailed"
        alpha = float(entry["Alpha Value"])

        if alpha not in critical_values[tail_type]:
            print(f"❌ No critical values defined for α = {alpha} and {tail_type}.")
            return

        crit_table = critical_values[tail_type][alpha]
        critical_val = crit_table.get(n)
        if critical_val is None:
            print(f"❌ No critical value found for n = {n} at α = {alpha}")
            return

        print(f"📉 Critical Value (α = {alpha}, {tail_type}) = {critical_val}")
        if T >= critical_val:
            print("❌ Reject H₀ — Significant result")
        else:
            print("✅ Fail to reject H₀ — Not significant")

    else:
        print("\nSTEP 5: Large Sample Test (Normal Approximation)")
        mean_T = n * (n + 1) / 4
        std_T = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
        alpha = float(entry['Alpha Value'])
        z = (T - mean_T) / std_T
        z_critical = norm.ppf(1 - alpha if tail in ['less', 'greater'] else 1 - alpha / 2)
        print(f"Mean T = {mean_T:.2f}, Std Dev = {std_T:.2f}")
        print(f"Z = {z:.3f}, Z Critical = ±{z_critical:.3f}")
        if (tail == 'less' and z < -z_critical) or \
           (tail == 'greater' and z > z_critical) or \
           (tail == 'two-sided' and abs(z) > z_critical):
            print("❌ Reject H₀ — Significant result")
        else:
            print("✅ Fail to reject H₀ — Not significant")

    print("\n✅ STEP 6: Final Decision — Test Completed.")
    return {
        "T": T,
        "T_plus": T_plus,
        "T_minus": T_minus,
        "diffs": diffs,
        "signed_ranks": signed_ranks,
        "n": n
    }

# ====== Main Execution ======
if __name__ == "__main__":
    file_path = str(DATA_DIR / "ARJ SIR DATASET.xlsx")
    structured_data = load_structured_data(file_path)

    print("Available Questions:")
    for entry in structured_data:
        print(f"Serial No {int(entry['Serial Number'])}: {entry['Question'][:80]}{'...' if len(entry['Question']) > 80 else ''}")

    try:
        serial_number_to_solve = int(input("\nEnter the Serial Number of the question to solve: "))
    except ValueError:
        print("❌ Invalid input. Please enter a valid integer.")
        exit()

    entry = next((e for e in structured_data if int(e["Serial Number"]) == serial_number_to_solve), None)
    if entry:
        print("\n" + "="*50)
        print(f"SOLVING QUESTION {serial_number_to_solve}")
        print("="*50)
        result = run_wilcoxon_test(entry)
    else:
        print(f"❌ Serial number {serial_number_to_solve} not found.")


In [ ]:
#MANN WHITNEY
import pandas as pd
import numpy as np
from scipy.stats import norm

# Load Z-critical value table from CSV
def load_z_table(file_path):
    z_df = pd.read_csv(file_path, index_col=0)
    z_df.columns = [str(col) for col in z_df.columns]
    z_df.index = [str(idx) for idx in z_df.index]
    return z_df

# Lookup Z-critical value from the table
def get_z_critical_from_table(alpha, z_table, tail_type='two-tailed'):
    if tail_type == 'two-tailed':
        prob = 0.5 - alpha / 2
    elif tail_type in ['one-tailed(left)', 'one-tailed(right)']:
        prob = 0.5 - alpha
    else:
        prob = 0.5 - alpha / 2

    flat_table = []
    for row in z_table.index:
        for col in z_table.columns:
            try:
                value = float(z_table.loc[row, col])
                z_val = float(row) + float(col)
                flat_table.append((value, z_val))
            except:
                continue

    flat_table.sort()  # Sort by probability

    for i in range(len(flat_table) - 1):
        low_prob, low_z = flat_table[i]
        high_prob, high_z = flat_table[i + 1]

        if low_prob <= prob <= high_prob:
            interpolated_z = low_z + (prob - low_prob) * (high_z - low_z) / (high_prob - low_prob)
            return round(interpolated_z, 3)

    from scipy.stats import norm
    print("⚠️ Prob not in Z-table range. Using scipy fallback.")
    return round(norm.ppf(1 - alpha / 2), 3)


# Load the p-critical values pivot-style table
def load_p_critical_lookup_table(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.astype(str)  # Ensure string headers
    return df

def get_p_critical(n1, n2, U, p_critical_df):
    try:
        # Ensure U and sample sizes are integers
        U = int(U)
        n1 = str(n1)  # ensure column name match

        if n1 not in p_critical_df.columns:
            print(f"❌ Column for n1={n1} not found in p-critical table.")
            return None

        # Filter by n2 and U using correct column names
        filtered = p_critical_df[(p_critical_df["n2"] == n2) & (p_critical_df["U"] == U)]

        if not filtered.empty:
            val = filtered.iloc[0][n1]
            if val == '-' or pd.isna(val):
                return None
            return float(val)
        else:
            print(f"⚠️ No match found for n2={n2}, U={U}")
            return None
    except Exception as e:
        print(f"❌ Error fetching p-critical value: {e}")
        return None


# Load and structure the data
def load_mannwhitney_data(file_path):
    df = pd.read_excel(file_path)
    structured_data = []
    i = 0
    while i < len(df):
        row = df.iloc[i]
        if not pd.isna(row[0]) and isinstance(row[0], (int, float)):
            serial_no = row[0]
            question_text = row[1]
            alpha_value = row[6] if len(row) > 6 and not pd.isna(row[6]) else None
            tail_type = row[7].strip().lower() if len(row) > 7 and not pd.isna(row[7]) else "two-tailed"
            data_cols = []
            for j in range(2, len(row)):
                if not pd.isna(row[j]) and isinstance(row[j], str):
                    data_cols.append((j, row[j]))
            data_rows = []
            next_row = i + 1
            while next_row < len(df) and pd.isna(df.iloc[next_row, 0]):
                data_row = {}
                for col_idx, col_name in data_cols:
                    if col_idx < len(df.columns) and not pd.isna(df.iloc[next_row, col_idx]):
                        data_row[col_name] = df.iloc[next_row, col_idx]
                if data_row:
                    data_rows.append(data_row)
                next_row += 1
            if data_rows:
                data_df = pd.DataFrame(data_rows)
                data_df = data_df.apply(pd.to_numeric, errors='ignore')
                structured_data.append({
                    'SR NO': serial_no,
                    'Question': question_text,
                    'Data': data_df,
                    'Alpha Value': alpha_value,
                    'Tail': tail_type
                })
            i = next_row - 1
        i += 1
    return structured_data

def map_tail_to_alternative(tail_type):
    if tail_type == "one-tailed(left)":
        return "less"
    elif tail_type == "one-tailed(right)":
        return "greater"
    else:
        return "two-sided"

def run_mannwhitney_test(entry, z_table, p_table):
    print(f"\n🔢 Serial No: {entry['SR NO']}")
    print(f"📘 Question: {entry['Question']}")
    print(f"🎯 Alpha Value: {entry['Alpha Value']}")
    print(f"🎯 Tail: {entry['Tail']}")

    data = entry['Data']
    numeric_cols = data.select_dtypes(include='number').columns.tolist()
    if len(numeric_cols) < 2:
        print("❌ Need at least two numeric columns for Mann-Whitney test.")
        return

    col1, col2 = numeric_cols[:2]
    group1 = data[col1].dropna().astype(float)
    group2 = data[col2].dropna().astype(float)

    print(f"\n📈 Groups Used: {col1} vs {col2}")
    print(f"📊 Sample Sizes: {len(group1)} vs {len(group2)}")

    tail_type = entry['Tail']
    tail = map_tail_to_alternative(tail_type)

    print("\nSTEP 1: Define Hypotheses")
    if tail == "two-sided":
        print("H₀: Group1 = Group2\nH₁: Group1 ≠ Group2")
    elif tail == "greater":
        print("H₀: Group1 ≤ Group2\nH₁: Group1 > Group2")
    elif tail == "less":
        print("H₀: Group1 ≥ Group2\nH₁: Group1 < Group2")

    print("\nSTEP 2: Combine and Rank Data")
    combined = np.concatenate([group1, group2])
    ranks = pd.Series(combined).rank()
    groups = ['Group 1'] * len(group1) + ['Group 2'] * len(group2)
    df = pd.DataFrame({'Value': combined, 'Rank': ranks, 'Group': groups})
    df = df.sort_values(by='Value')
    print(df.to_string(index=False))

    print("\nSTEP 3: Mann-Whitney U Test")

    # Ensure n1 <= n2
    if len(group1) <= len(group2):
        g1, g2 = group1, group2
    else:
        g1, g2 = group2, group1

    n1, n2 = len(g1), len(g2)
    alpha_raw = entry['Alpha Value']
    alpha = 0.05 if pd.isna(alpha_raw) or str(alpha_raw).strip().lower() == "not given" else float(alpha_raw)

    R1 = sum(pd.Series(np.concatenate([g1, g2])).rank()[:n1])
    U1 = R1 - (n1 * (n1 + 1)) / 2
    U2 = n1 * n2 - U1
    U = min(U1, U2)
    print(f"U1 = {U1:.3f}, U2 = {U2:.3f}, Using U = {U:.3f}")

    if n1 <= 10 and n2 <= 10:
        p_critical = get_p_critical(n1, n2, int(U), p_table)
        if p_critical is None:
            print("⚠️ p-critical value not found for this configuration.")
            return
        print(f"P-critical = {p_critical}, Alpha = {alpha}")
        if p_critical < alpha:
            print("❌ Reject H₀ — Significant result")
        else:
            print("✅ Fail to reject H₀ — Not significant")
    else:
        mean_U = (n1 * n2) / 2
        var_U = (n1 * n2 * (n1 + n2 + 1)) / 12
        z_score = (U - mean_U) / np.sqrt(var_U)
        z_critical = get_z_critical_from_table(alpha, z_table, tail_type)

        print(f"Mean of U = {mean_U:.3f}")
        print(f"Variance of U = {var_U:.3f}")
        print(f"Z-score = {z_score:.3f}")
        if tail_type == "two-tailed":
            print(f"Z-critical = ±{z_critical}")
        elif tail_type == "one-tailed(left)":
            print(f"Z-critical = { -z_critical } (Left-tailed)")
        elif tail_type == "one-tailed(right)":
            print(f"Z-critical = { z_critical } (Right-tailed)")

        if tail_type == "two-tailed":
            if abs(z_score) > z_critical:
                print("❌ Reject H₀ — Significant difference")
            else:
                print("✅ Fail to reject H₀ — Not significant")
        elif tail_type == "one-tailed(left)":
            if z_score < -z_critical:
                print("❌ Reject H₀ — Significant (left tail)")
            else:
                print("✅ Fail to reject H₀ — Not significant")
        elif tail_type == "one-tailed(right)":
            if z_score > z_critical:
                print("❌ Reject H₀ — Significant (right tail)")
            else:
                print("✅ Fail to reject H₀ — Not significant")

    print("\n✅ STEP 4: Final Decision — Test Completed.")

# === Main execution ===
mannwhitney_file = "/content/ARJ SIR DATASET (1).xlsx"
z_table_file = "/content/z_table_complete (1).csv"
p_table_file = "//content/p_critical_values (1).csv"

structured_data = load_mannwhitney_data(mannwhitney_file)
z_table = load_z_table(z_table_file)
p_table = load_p_critical_lookup_table(p_table_file) # Changed function call to load_p_critical_lookup_table

# Show available questions
print("Available questions:")
for entry in structured_data:
    print(f"Serial No: {entry['SR NO']}: {entry['Question'][:80]}...")

# Take user input for serial number
serial_number_to_solve = int(input("\nEnter the Serial Number of the question to solve: "))  # Dynamic input

# Find and run the Mann-Whitney test for the selected question
entry = next((e for e in structured_data if int(e["SR NO"]) == serial_number_to_solve), None)
if entry:
    print("\n" + "="*50)
    print(f"SOLVING QUESTION {serial_number_to_solve}")
    print("="*50)
    run_mannwhitney_test(entry, z_table, p_table)
else:
    print(f"❌ Serial number {serial_number_to_solve} not found.")


In [ ]:

#CHUMESWARI
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Step 1: Load and prepare the training dataset
train_df = pd.read_csv(str(DATA_DIR / "Non_Parametric_combined.csv"), header=None, names=["Question", "Label"])
X = train_df["Question"].fillna('')
y = train_df["Label"].fillna(train_df["Label"].mode()[0])

# Step 2: Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=16)

# Step 3: TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Step 4: Train model
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_tfidf, y_train)

# Step 5: Model Evaluation
y_pred = dt_model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Step 6: Load Excel file for SR NO-based question prediction
excel_df = pd.read_excel(str(DATA_DIR / "ARJ SIR DATASET.xlsx"))
excel_df['SR NO'] = excel_df['SR NO'].ffill()
excel_df['Question'] = excel_df['Question'].ffill()
questions_df = excel_df.groupby('SR NO').first().reset_index()

# Step 7: Function to predict test type from SR NO
def predict_test_by_sr_no(sr_no):
    try:
        sr_no = float(sr_no)
        question = questions_df.loc[questions_df['SR NO'] == sr_no, 'Question'].values[0]
        question_tfidf = vectorizer.transform([question])
        predicted_label = dt_model.predict(question_tfidf)[0]
        return f"Predicted test for SR NO {int(sr_no)}: {predicted_label}"
    except IndexError:
        return "SR NO not found in the dataset."
    except Exception as e:
        return f"Error: {str(e)}"

# Step 8: Use it
sr_no_input = input("Enter SR NO: ")
prediction = predict_test_by_sr_no(sr_no_input)
print(prediction)

if "Runs Test" in prediction:
    #RUNS
    import pandas as pd
    import numpy as np
    import re

    # Load the datasets (update paths if needed)
    df = pd.read_excel(str(DATA_DIR / "ARJ SIR DATASET.xlsx"))
    critical_df = pd.read_csv(str(DATA_DIR / "runs_test_critical_values_complete.csv"))
    z_table_df = pd.read_csv(str(DATA_DIR / "z_table_complete.csv"))

    # ✅ Clean column names
    df.columns = df.columns.str.strip()
    critical_df.columns = critical_df.columns.str.strip()
    z_table_df.columns = z_table_df.columns.str.strip()

    def extract_sequence(sequence_text):
        sequence_text = sequence_text.strip()
        numbers = re.findall(r'\d+\.?\d*', sequence_text)
        if numbers:
            numbers = list(map(float, numbers))
            return ['H' if num > 7 else 'L' for num in numbers]
        categorical = re.findall(r'[A-Za-z]', sequence_text)
        if categorical:
            return [char.upper() for char in categorical]
        return None

    def count_runs(sequence):
        runs = 1
        for i in range(1, len(sequence)):
            if sequence[i] != sequence[i - 1]:
                runs += 1
        return runs

    def get_critical_values(n1, n2):
        try:
            col_name = str(n2)
            if n1 not in critical_df['n1/n2'].values:
                return None, None
            lower_row = critical_df[(critical_df['n1/n2'] == n1) & (critical_df['Table'] == 'Lower Tail')]
            upper_row = critical_df[(critical_df['n1/n2'] == n1) & (critical_df['Table'] == 'Upper Tail')]
            if lower_row.empty or upper_row.empty or col_name not in critical_df.columns:
                return None, None
            R_low = int(lower_row[col_name].values[0])
            R_high = int(upper_row[col_name].values[0])
            return R_low, R_high
        except:
            return None, None

    def get_z_critical(alpha):
        try:
            alpha_half = alpha / 2
            target_prob = 0.5 - alpha_half
            closest_diff = float('inf')
            closest_z = None
            for row_z in z_table_df['z']:
                for col in z_table_df.columns[1:]:
                    prob = z_table_df.loc[z_table_df['z'] == row_z, col].values[0]
                    if isinstance(prob, str):
                        prob = float(prob)
                    diff = abs(prob - target_prob)
                    if diff < closest_diff:
                        closest_diff = diff
                        closest_z = float(row_z) + float(col)
            return closest_z
        except:
            return None

    def runs_test_solver(sr_no):
        row = df[df['SR NO'] == sr_no]
        if row.empty:
            print(f"❌ SR NO {sr_no} not found.")
            return
        question_text = row['Question'].values[0]
        sequence_text = row['Data'].values[0]
        alpha_value = row['Alpha value'].values[0]

        print(f"\n🔹 SR NO {sr_no}: {question_text}")
        sequence = extract_sequence(sequence_text)
        if not sequence or len(set(sequence)) != 2:
            print("❌ Error: Invalid or non-binary sequence.")
            return

        unique_vals = list(set(sequence))
        n1 = sequence.count(unique_vals[0])
        n2 = sequence.count(unique_vals[1])
        R = count_runs(sequence)

        print(f"📊 n1 ({unique_vals[0]}) = {n1}, n2 ({unique_vals[1]}) = {n2}")
        print(f"🔢 Observed Runs (R) = {R}")

        if n1 + n2 <= 20:
            print("📏 Small Sample Test")
            R_low, R_high = get_critical_values(n1, n2)
            if R_low is None or R_high is None:
                print("❌ Critical values not found.")
                return
            print(f"📉 Critical Values: R_low = {R_low}, R_high = {R_high}")
            if R < R_low or R > R_high:
                print("❌ Reject H0 (Not random)")
            else:
                print("✅ Fail to reject H0 (Random)")
        else:
            print("📏 Large Sample Test")
            mean_R = ((2 * n1 * n2) / (n1 + n2)) + 1
            std_dev_R = np.sqrt((2 * n1 * n2 * (2 * n1 * n2 - n1 - n2)) / (((n1 + n2)**2) * (n1 + n2 - 1)))
            z = (R - mean_R) / std_dev_R
            z_critical = get_z_critical(float(alpha_value))

            print(f"📊 Mean (μR) = {mean_R:.3f}")
            print(f"📉 Std Dev (σR) = {std_dev_R:.3f}")
            print(f"📈 Z = {z:.3f}, Z_critical = ±{z_critical:.3f}")
            if abs(z) > abs(z_critical):
                print("❌ Reject H0 (Not random)")
            else:
                print("✅ Fail to reject H0 (Random)")

    # 🔹 Take user input for SR NO
    try:
        # sr_no_input = int(input("Enter SR NO to run the Runs Test: "))
        sr_no_input = int(sr_no_input)
        runs_test_solver(sr_no_input)
    except ValueError:
        print("❌ Invalid input. Please enter a valid SR NO (integer).")

if "Wilcoxon test" in prediction:
    #WILCOXON
    import pandas as pd
    import numpy as np
    from scipy.stats import norm

    # Load and structure the data
    def load_structured_data(file_path):
        # Use pd.read_excel to read Excel files
        df = pd.read_excel(file_path)
        structured_data = []

        i = 0
        while i < len(df):
            row = df.iloc[i]
            if not pd.isna(row[0]) and isinstance(row[0], (int, float)):
                serial_no = row[0]
                question_text = row[1]
                alpha_value = row[6] if len(row) > 6 and not pd.isna(row[6]) else None
                data_cols = []
                for j in range(2, len(row)):
                    if not pd.isna(row[j]) and isinstance(row[j], str):
                        data_cols.append((j, row[j]))
                data_rows = []
                next_row = i + 1
                while next_row < len(df) and pd.isna(df.iloc[next_row, 0]):
                    data_row = {}
                    for col_idx, col_name in data_cols:
                        if col_idx < len(df.columns) and not pd.isna(df.iloc[next_row, col_idx]):
                            data_row[col_name] = df.iloc[next_row, col_idx]
                    if data_row:
                        data_rows.append(data_row)
                    next_row += 1
                if data_rows:
                    data_df = pd.DataFrame(data_rows)
                    data_df = data_df.apply(pd.to_numeric, errors='ignore')
                    structured_data.append({
                        'Serial Number': serial_no,
                        'Question': question_text,
                        'Data': data_df,
                        'Alpha Value': alpha_value
                    })
                i = next_row - 1
            i += 1
        return structured_data

    def infer_hypotheses(question_text):
        q = question_text.lower()
        if "greater than" in q or "at least" in q:
            return 0, "greater", "H₀: Median ≤ baseline\nH₁: Median > baseline"
        elif "less than" in q:
            return 0, "less", "H₀: Median ≥ baseline\nH₁: Median < baseline"
        elif "different" in q or "difference" in q or "two-sided" in q:
            return 0, "two-sided", "H₀: Median = baseline\nH₁: Median ≠ baseline"
        elif "effective" in q or "not effective" in q:
            return 0, "two-sided", "H₀: Not effective\nH₁: Effective"
        else:
            return 0, "two-sided", "H₀: No effect\nH₁: There is an effect"

    def paired_signed_rank_test(data1, data2):
        diffs = [d1 - d2 for d1, d2 in zip(data1, data2)]
        non_zero = [(i, d) for i, d in enumerate(diffs) if d != 0]
        if not non_zero:
            raise ValueError("All paired differences are zero.")

        abs_diff = [(i, abs(d)) for i, d in non_zero]
        abs_diff.sort(key=lambda x: x[1])
        ranks = [0] * len(diffs)
        i = 0
        current_rank = 1
        while i < len(abs_diff):
            tie_val = abs_diff[i][1]
            tie_group = [abs_diff[i]]
            j = i + 1
            while j < len(abs_diff) and abs_diff[j][1] == tie_val:
                tie_group.append(abs_diff[j])
                j += 1
            avg_rank = sum(range(current_rank, current_rank + len(tie_group))) / len(tie_group)
            for idx, _ in tie_group:
                ranks[idx] = avg_rank
            current_rank += len(tie_group)
            i = j
        signed_ranks = [np.sign(diffs[i]) * ranks[i] for i in range(len(diffs))]
        T_plus = sum(r for r in signed_ranks if r > 0)
        T_minus = sum(-r for r in signed_ranks if r < 0)
        T = min(T_plus, T_minus)
        return diffs, signed_ranks, T, T_plus, T_minus

    def signed_rank_test(data, median):
        diffs = [x - median for x in data]
        non_zero = [(i, d) for i, d in enumerate(diffs) if d != 0]
        if not non_zero:
            raise ValueError("All values equal to the hypothesized median.")
        abs_diff = [(i, abs(d)) for i, d in non_zero]
        abs_diff.sort(key=lambda x: x[1])
        ranks = [0] * len(diffs)
        i = 0
        current_rank = 1
        while i < len(abs_diff):
            tie_val = abs_diff[i][1]
            tie_group = [abs_diff[i]]
            j = i + 1
            while j < len(abs_diff) and abs_diff[j][1] == tie_val:
                tie_group.append(abs_diff[j])
                j += 1
            avg_rank = sum(range(current_rank, current_rank + len(tie_group))) / len(tie_group)
            for idx, _ in tie_group:
                ranks[idx] = avg_rank
            current_rank += len(tie_group)
            i = j
        signed_ranks = [np.sign(diffs[i]) * ranks[i] for i in range(len(diffs))]
        T_plus = sum(r for r in signed_ranks if r > 0)
        T_minus = sum(-r for r in signed_ranks if r < 0)
        T = min(T_plus, T_minus)
        return diffs, signed_ranks, T, T_plus, T_minus

    def run_wilcoxon_test(entry):
        print(f"\n🔢 Serial No: {entry['Serial Number']}")
        print(f"📘 Question: {entry['Question']}")
        print(f"🎯 Overriding Alpha Value to 0.05")
        entry['Alpha Value'] = 0.05

        data = entry['Data']
        numeric_cols = data.select_dtypes(include='number').columns.tolist()
        if not numeric_cols:
            print("❌ No numeric columns found.")
            return

        if len(numeric_cols) == 3:
            print(f"\n📈 Using columns: {numeric_cols[1]} and {numeric_cols[2]} for paired test")
            values1 = data[numeric_cols[1]].dropna().astype(float).tolist()
            values2 = data[numeric_cols[2]].dropna().astype(float).tolist()
            min_len = min(len(values1), len(values2))
            values1 = values1[:min_len]
            values2 = values2[:min_len]
            print(f"📊 Sample Size: {min_len} pairs")
            median, tail, hypotheses = infer_hypotheses(entry['Question'])
            hypotheses = "H₀: No difference between paired samples\nH₁: There is a difference between paired samples"
            if "greater" in tail:
                hypotheses = "H₀: Second sample ≤ First sample\nH₁: Second sample > First sample"
            elif "less" in tail:
                hypotheses = "H₀: Second sample ≥ First sample\nH₁: Second sample < First sample"
            print("\nSTEP 1: Define Hypotheses")
            print(hypotheses)
            print("\nSTEP 2: Wilcoxon Signed-Rank Test (Paired)")
            diffs, signed_ranks, T, T_plus, T_minus = paired_signed_rank_test(values1, values2)
        else:
            test_col = numeric_cols[-1]
            values = data[test_col].dropna().astype(float).tolist()
            print(f"\n📈 Data Column Used: {test_col}")
            print(f"📊 Sample Size: {len(values)}")
            baseline_val = 0
            if "8 days" in entry['Question']:
                baseline_val = 8
            elif "5 days" in entry['Question']:
                baseline_val = 5
            median, tail, hypotheses = infer_hypotheses(entry['Question'])
            median = baseline_val
            print("\nSTEP 1: Define Hypotheses")
            print(hypotheses)
            print("\nSTEP 2: Wilcoxon Signed-Rank Test (One Sample)")
            diffs, signed_ranks, T, T_plus, T_minus = signed_rank_test(values, median)

        n = sum(1 for d in diffs if d != 0)
        print("\nSTEP 3: Differences and Signed Ranks")
        for i, (diff, rank) in enumerate(zip(diffs, signed_ranks)):
            if diff != 0:
                print(f"Pair {i+1}: Diff = {diff:.2f}, Rank = {rank:.2f}")
        print(f"\nSTEP 4: T+ = {T_plus:.2f}, T- = {T_minus:.2f}")
        print(f"Test Statistic (T) = min(T+, T-) = {T:.2f}")

        if n <= 25:
            print("\nSTEP 5: Small Sample Test")
            alpha = float(entry['Alpha Value'])

                    # Expanded critical values for small samples and multiple alpha levels
            critical_values = {
                "one-tailed": {
                    0.10: {
                        5: 0, 6: 2, 7: 3, 8: 5, 9: 7, 10: 10, 11: 13, 12: 16, 13: 20,
                        14: 24, 15: 29, 16: 34, 17: 39, 18: 45, 19: 51, 20: 58, 21: 65,
                        22: 72, 23: 80, 24: 88, 25: 96
                    },
                    0.05: {
                        5: 0, 6: 2, 7: 3, 8: 6, 9: 8, 10: 11, 11: 14, 12: 17, 13: 21,
                        14: 26, 15: 30, 16: 36, 17: 41, 18: 47, 19: 54, 20: 60, 21: 68,
                        22: 75, 23: 83, 24: 92, 25: 101
                    },
                    0.025: {
                        6: 0, 7: 1, 8: 3, 9: 5, 10: 8, 11: 11, 12: 14, 13: 18, 14: 23,
                        15: 28, 16: 33, 17: 39, 18: 45, 19: 51, 20: 58, 21: 66, 22: 74,
                        23: 82, 24: 91, 25: 100
                    },
                    0.01: {
                        6: 0, 7: 1, 8: 2, 9: 4, 10: 6, 11: 9, 12: 12, 13: 16, 14: 21,
                        15: 26, 16: 31, 17: 37, 18: 43, 19: 49, 20: 56, 21: 64, 22: 72,
                        23: 81, 24: 90, 25: 99
                    }
                },
                "two-tailed": {
                    0.10: {
                        5: 0, 6: 1, 7: 2, 8: 4, 9: 6, 10: 9, 11: 12, 12: 15, 13: 19,
                        14: 24, 15: 28, 16: 34, 17: 39, 18: 45, 19: 51, 20: 58, 21: 66,
                        22: 73, 23: 81, 24: 90, 25: 99
                    },
                    0.05: {
                        5: 0, 6: 2, 7: 4, 8: 6, 9: 8, 10: 11, 11: 14, 12: 17, 13: 21,
                        14: 26, 15: 30, 16: 36, 17: 41, 18: 47, 19: 54, 20: 60, 21: 68,
                        22: 75, 23: 83, 24: 92, 25: 101
                    },
                    0.025: {
                        6: 0, 7: 1, 8: 3, 9: 5, 10: 8, 11: 11, 12: 14, 13: 18, 14: 23,
                        15: 28, 16: 33, 17: 39, 18: 45, 19: 51, 20: 58, 21: 66, 22: 74,
                        23: 82, 24: 91, 25: 100
                    },
                    0.01: {
                        6: 0, 7: 1, 8: 2, 9: 4, 10: 6, 11: 9, 12: 12, 13: 16, 14: 21,
                        15: 26, 16: 31, 17: 37, 18: 43, 19: 49, 20: 56, 21: 64, 22: 72,
                        23: 81, 24: 90, 25: 99
                    }
                }
            }

            tail_type = "two-tailed" if tail == "two-sided" else "one-tailed"
            alpha = float(entry["Alpha Value"])

            if alpha not in critical_values[tail_type]:
                print(f"❌ No critical values defined for α = {alpha} and {tail_type}.")
                return

            crit_table = critical_values[tail_type][alpha]
            critical_val = crit_table.get(n)
            if critical_val is None:
                print(f"❌ No critical value found for n = {n} at α = {alpha}")
                return

            print(f"📉 Critical Value (α = {alpha}, {tail_type}) = {critical_val}")
            if T >= critical_val:
                print("❌ Reject H₀ — Significant result")
            else:
                print("✅ Fail to reject H₀ — Not significant")

        else:
            print("\nSTEP 5: Large Sample Test (Normal Approximation)")
            mean_T = n * (n + 1) / 4
            std_T = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
            alpha = float(entry['Alpha Value'])
            z = (T - mean_T) / std_T
            z_critical = norm.ppf(1 - alpha if tail in ['less', 'greater'] else 1 - alpha / 2)
            print(f"Mean T = {mean_T:.2f}, Std Dev = {std_T:.2f}")
            print(f"Z = {z:.3f}, Z Critical = ±{z_critical:.3f}")
            if (tail == 'less' and z < -z_critical) or \
              (tail == 'greater' and z > z_critical) or \
              (tail == 'two-sided' and abs(z) > z_critical):
                print("❌ Reject H₀ — Significant result")
            else:
                print("✅ Fail to reject H₀ — Not significant")

        print("\n✅ STEP 6: Final Decision — Test Completed.")
        return {
            "T": T,
            "T_plus": T_plus,
            "T_minus": T_minus,
            "diffs": diffs,
            "signed_ranks": signed_ranks,
            "n": n
        }

    # ====== Main Execution ======
    if __name__ == "__main__":
        file_path = str(DATA_DIR / "ARJ SIR DATASET.xlsx")
        structured_data = load_structured_data(file_path)

        print("Available Questions:")
        for entry in structured_data:
            print(f"Serial No {int(entry['Serial Number'])}: {entry['Question'][:80]}{'...' if len(entry['Question']) > 80 else ''}")

        try:
            # serial_number_to_solve = int(input("\nEnter the Serial Number of the question to solve: "))
            serial_number_to_solve = int(sr_no_input)
        except ValueError:
            print("❌ Invalid input. Please enter a valid integer.")
            exit()

        entry = next((e for e in structured_data if int(e["Serial Number"]) == serial_number_to_solve), None)
        if entry:
            print("\n" + "="*50)
            print(f"SOLVING QUESTION {serial_number_to_solve}")
            print("="*50)
            result = run_wilcoxon_test(entry)
        else:
            print(f"❌ Serial number {serial_number_to_solve} not found.")

if "Mann Whitney Test" in prediction:
    #MANN WHITNEY
    import pandas as pd
    import numpy as np
    from scipy.stats import norm

    # Load Z-critical value table from CSV
    def load_z_table(file_path):
        z_df = pd.read_csv(file_path, index_col=0)
        z_df.columns = [str(col) for col in z_df.columns]
        z_df.index = [str(idx) for idx in z_df.index]
        return z_df

    # Lookup Z-critical value from the table
    def get_z_critical_from_table(alpha, z_table, tail_type='two-tailed'):
        if tail_type == 'two-tailed':
            prob = 0.5 - alpha / 2
        elif tail_type in ['one-tailed(left)', 'one-tailed(right)']:
            prob = 0.5 - alpha
        else:
            prob = 0.5 - alpha / 2

        flat_table = []
        for row in z_table.index:
            for col in z_table.columns:
                try:
                    value = float(z_table.loc[row, col])
                    z_val = float(row) + float(col)
                    flat_table.append((value, z_val))
                except:
                    continue

        flat_table.sort()  # Sort by probability

        for i in range(len(flat_table) - 1):
            low_prob, low_z = flat_table[i]
            high_prob, high_z = flat_table[i + 1]

            if low_prob <= prob <= high_prob:
                interpolated_z = low_z + (prob - low_prob) * (high_z - low_z) / (high_prob - low_prob)
                return round(interpolated_z, 3)

        from scipy.stats import norm
        print("⚠️ Prob not in Z-table range. Using scipy fallback.")
        return round(norm.ppf(1 - alpha / 2), 3)


    # Load the p-critical values pivot-style table
    def load_p_critical_lookup_table(file_path):
        df = pd.read_csv(file_path)
        df.columns = df.columns.astype(str)  # Ensure string headers
        return df

    def get_p_critical(n1, n2, U, p_critical_df):
        try:
            # Ensure U and sample sizes are integers
            U = int(U)
            n1 = str(n1)  # ensure column name match

            if n1 not in p_critical_df.columns:
                print(f"❌ Column for n1={n1} not found in p-critical table.")
                return None

            # Filter by n2 and U using correct column names
            filtered = p_critical_df[(p_critical_df["n2"] == n2) & (p_critical_df["U"] == U)]

            if not filtered.empty:
                val = filtered.iloc[0][n1]
                if val == '-' or pd.isna(val):
                    return None
                return float(val)
            else:
                print(f"⚠️ No match found for n2={n2}, U={U}")
                return None
        except Exception as e:
            print(f"❌ Error fetching p-critical value: {e}")
            return None


    # Load and structure the data
    def load_mannwhitney_data(file_path):
        df = pd.read_excel(file_path)
        structured_data = []
        i = 0
        while i < len(df):
            row = df.iloc[i]
            if not pd.isna(row[0]) and isinstance(row[0], (int, float)):
                serial_no = row[0]
                question_text = row[1]
                alpha_value = row[6] if len(row) > 6 and not pd.isna(row[6]) else None
                tail_type = row[7].strip().lower() if len(row) > 7 and not pd.isna(row[7]) else "two-tailed"
                data_cols = []
                for j in range(2, len(row)):
                    if not pd.isna(row[j]) and isinstance(row[j], str):
                        data_cols.append((j, row[j]))
                data_rows = []
                next_row = i + 1
                while next_row < len(df) and pd.isna(df.iloc[next_row, 0]):
                    data_row = {}
                    for col_idx, col_name in data_cols:
                        if col_idx < len(df.columns) and not pd.isna(df.iloc[next_row, col_idx]):
                            data_row[col_name] = df.iloc[next_row, col_idx]
                    if data_row:
                        data_rows.append(data_row)
                    next_row += 1
                if data_rows:
                    data_df = pd.DataFrame(data_rows)
                    data_df = data_df.apply(pd.to_numeric, errors='ignore')
                    structured_data.append({
                        'SR NO': serial_no,
                        'Question': question_text,
                        'Data': data_df,
                        'Alpha Value': alpha_value,
                        'Tail': tail_type
                    })
                i = next_row - 1
            i += 1
        return structured_data

    def map_tail_to_alternative(tail_type):
        if tail_type == "one-tailed(left)":
            return "less"
        elif tail_type == "one-tailed(right)":
            return "greater"
        else:
            return "two-sided"

    def run_mannwhitney_test(entry, z_table, p_table):
        print(f"\n🔢 Serial No: {entry['SR NO']}")
        print(f"📘 Question: {entry['Question']}")
        print(f"🎯 Alpha Value: {entry['Alpha Value']}")
        print(f"🎯 Tail: {entry['Tail']}")

        data = entry['Data']
        numeric_cols = data.select_dtypes(include='number').columns.tolist()
        if len(numeric_cols) < 2:
            print("❌ Need at least two numeric columns for Mann-Whitney test.")
            return

        col1, col2 = numeric_cols[:2]
        group1 = data[col1].dropna().astype(float)
        group2 = data[col2].dropna().astype(float)

        print(f"\n📈 Groups Used: {col1} vs {col2}")
        print(f"📊 Sample Sizes: {len(group1)} vs {len(group2)}")

        tail_type = entry['Tail']
        tail = map_tail_to_alternative(tail_type)

        print("\nSTEP 1: Define Hypotheses")
        if tail == "two-sided":
            print("H₀: Group1 = Group2\nH₁: Group1 ≠ Group2")
        elif tail == "greater":
            print("H₀: Group1 ≤ Group2\nH₁: Group1 > Group2")
        elif tail == "less":
            print("H₀: Group1 ≥ Group2\nH₁: Group1 < Group2")

        print("\nSTEP 2: Combine and Rank Data")
        combined = np.concatenate([group1, group2])
        ranks = pd.Series(combined).rank()
        groups = ['Group 1'] * len(group1) + ['Group 2'] * len(group2)
        df = pd.DataFrame({'Value': combined, 'Rank': ranks, 'Group': groups})
        df = df.sort_values(by='Value')
        print(df.to_string(index=False))

        print("\nSTEP 3: Mann-Whitney U Test")

        # Ensure n1 <= n2
        if len(group1) <= len(group2):
            g1, g2 = group1, group2
        else:
            g1, g2 = group2, group1

        n1, n2 = len(g1), len(g2)
        alpha_raw = entry['Alpha Value']
        alpha = 0.05 if pd.isna(alpha_raw) or str(alpha_raw).strip().lower() == "not given" else float(alpha_raw)

        R1 = sum(pd.Series(np.concatenate([g1, g2])).rank()[:n1])
        U1 = R1 - (n1 * (n1 + 1)) / 2
        U2 = n1 * n2 - U1
        U = min(U1, U2)
        print(f"U1 = {U1:.3f}, U2 = {U2:.3f}, Using U = {U:.3f}")

        if n1 <= 10 and n2 <= 10:
            p_critical = get_p_critical(n1, n2, int(U), p_table)
            if p_critical is None:
                print("⚠️ p-critical value not found for this configuration.")
                return
            print(f"P-critical = {p_critical}, Alpha = {alpha}")
            if p_critical < alpha:
                print("❌ Reject H₀ — Significant result")
            else:
                print("✅ Fail to reject H₀ — Not significant")
        else:
            mean_U = (n1 * n2) / 2
            var_U = (n1 * n2 * (n1 + n2 + 1)) / 12
            z_score = (U - mean_U) / np.sqrt(var_U)
            z_critical = get_z_critical_from_table(alpha, z_table, tail_type)

            print(f"Mean of U = {mean_U:.3f}")
            print(f"Variance of U = {var_U:.3f}")
            print(f"Z-score = {z_score:.3f}")
            if tail_type == "two-tailed":
                print(f"Z-critical = ±{z_critical}")
            elif tail_type == "one-tailed(left)":
                print(f"Z-critical = { -z_critical } (Left-tailed)")
            elif tail_type == "one-tailed(right)":
                print(f"Z-critical = { z_critical } (Right-tailed)")

            if tail_type == "two-tailed":
                if abs(z_score) > z_critical:
                    print("❌ Reject H₀ — Significant difference")
                else:
                    print("✅ Fail to reject H₀ — Not significant")
            elif tail_type == "one-tailed(left)":
                if z_score < -z_critical:
                    print("❌ Reject H₀ — Significant (left tail)")
                else:
                    print("✅ Fail to reject H₀ — Not significant")
            elif tail_type == "one-tailed(right)":
                if z_score > z_critical:
                    print("❌ Reject H₀ — Significant (right tail)")
                else:
                    print("✅ Fail to reject H₀ — Not significant")

        print("\n✅ STEP 4: Final Decision — Test Completed.")

    # === Main execution ===
    mannwhitney_file = str(DATA_DIR / "ARJ SIR DATASET.xlsx")
    z_table_file = str(DATA_DIR / "z_table_complete.csv")
    p_table_file = str(DATA_DIR / "p_critical_values.csv")

    structured_data = load_mannwhitney_data(mannwhitney_file)
    z_table = load_z_table(z_table_file)
    p_table = load_p_critical_lookup_table(p_table_file) # Changed function call to load_p_critical_lookup_table

    # Show available questions
    print("Available questions:")
    for entry in structured_data:
        print(f"Serial No: {entry['SR NO']}: {entry['Question'][:80]}...")

    # Take user input for serial number
    # serial_number_to_solve = int(input("\nEnter the Serial Number of the question to solve: "))  # Dynamic input
    serial_number_to_solve = int(sr_no_input)

    # Find and run the Mann-Whitney test for the selected question
    entry = next((e for e in structured_data if int(e["SR NO"]) == serial_number_to_solve), None)
    if entry:
        print("\n" + "="*50)
        print(f"SOLVING QUESTION {serial_number_to_solve}")
        print("="*50)
        run_mannwhitney_test(entry, z_table, p_table)
    else:
        print(f"❌ Serial number {serial_number_to_solve} not found.")